#  Enterprise Performance Optimization & Maintenance

| Section | Topic |
|---|---|
| 1 | Setup & Configuration |
| 2 | Build Optimized Tables — Liquid Clustering vs Partitioned + Z-Order |
| 3 | Benchmarking — 5 query patterns, 3 runs each |
| 4 | MERGE INTO — Late-Arriving Data Upserts |
| 5 | Churn Metrics — Stale Inventory & Brand Activity |
| 6 | Delta Maintenance — VACUUM + OPTIMIZE |
| 7 | Final Summary & Strategy Comparison |

## Section 1 — Setup & Configuration

In [0]:
import time
from pyspark.sql import functions as F

dbutils.widgets.text('project_catalog', 'vstone_catalog')
dbutils.widgets.text('gold_schema',     'gold')
dbutils.widgets.text('silver_schema',   'silver')

CATALOG     = dbutils.widgets.get('project_catalog')
GOLD        = dbutils.widgets.get('gold_schema')
SILVER      = dbutils.widgets.get('silver_schema')

FACT        = f'{CATALOG}.{GOLD}.fact_listings'
FACT_LIQUID = f'{CATALOG}.{GOLD}.fact_listings_liquid'
FACT_PART   = f'{CATALOG}.{GOLD}.fact_listings_partitioned'
BENCH_TABLE = f'{CATALOG}.{GOLD}.benchmark_results'
CHURN_TABLE = f'{CATALOG}.{GOLD}.agg_stale_inventory'
BRAND_CHURN = f'{CATALOG}.{GOLD}.agg_brand_churn_metrics'

print('=' * 70)
print('  PERFORMANCE OPTIMIZATION SUITE — VStone Gold Layer')
print('=' * 70)
print(f'  Source fact table  : {FACT}')
print(f'  Liquid table       : {FACT_LIQUID}')
print(f'  Partitioned table  : {FACT_PART}')
print(f'  Benchmark results  : {BENCH_TABLE}')
print(f'  Churn table        : {CHURN_TABLE}')
print(f'  Brand churn        : {BRAND_CHURN}')
print('=' * 70)

fact_rows = spark.table(FACT).count()
print(f'\n  fact_listings rows: {fact_rows:,}')
print('  Ready.')

## Section 2 — Build Optimized Tables

### Strategy A: Liquid Clustering (Delta 3.0+ Recommended)


### Strategy B: Traditional Partitioning + Z-Order (Classic)


In [0]:
# ── APPROACH A: LIQUID CLUSTERING ────────────────────────────────────────────
print('━' * 70)
print('  APPROACH A: Liquid Clustering')
print('━' * 70)

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {FACT_LIQUID}
USING DELTA
CLUSTER BY (brand, listing_year, location_key, fuel_type)
TBLPROPERTIES (
  'quality'                    = 'silver',
  'optimization'               = 'liquid_clustering',
  'cluster_keys'               = 'brand_listing_year_location_key_fuel_type',
  'delta.enableChangeDataFeed' = 'true'
)
AS SELECT * FROM {FACT}
''')

spark.sql(f'OPTIMIZE {FACT_LIQUID}')
liq_rows = spark.table(FACT_LIQUID).count()
print(f'  Liquid table ready — {liq_rows:,} rows')
print('  Cluster keys: brand | listing_year | location_key | fuel_type')

In [0]:
# ── APPROACH B: PARTITIONING + Z-ORDER ───────────────────────────────────────
print('━' * 70)
print('  APPROACH B: Partitioning + Z-Order')
print('━' * 70)

spark.sql(f'''
CREATE TABLE IF NOT EXISTS {FACT_PART}
USING DELTA
PARTITIONED BY (fuel_type)
TBLPROPERTIES (
  'quality'                    = 'gold',
  'optimization'               = 'partition_zorder',
  'partition_key'              = 'fuel_type',
  'zorder_keys'                = 'brand_listing_year',
  'delta.enableChangeDataFeed' = 'true'
)
AS SELECT * FROM {FACT}
''')

# Z-Order hamesha table create hone ke baad separate OPTIMIZE command se hota hai
spark.sql(f'OPTIMIZE {FACT_PART} ZORDER BY (brand, listing_year)')

part_rows = spark.table(FACT_PART).count()
print(f'  Partitioned table ready — {part_rows:,} rows')
print('  Partition: fuel_type  |  Z-Order: brand, listing_year')

print('\n  Partition distribution (fuel_type):')
# F.desc use karne ke liye 'from pyspark.sql import functions as F' zaroori hai
spark.table(FACT_PART).groupBy('fuel_type').count().orderBy(F.desc('count')).show(10, truncate=False)

## Section 3 — Benchmarking Framework

**Method:** Each query runs 3 times. Average execution time is recorded.

| Query | Pattern | Why it tests |
|---|---|---|
| Q1 | Single brand filter | High-cardinality column |
| Q2 | Multi-brand + mileage range | In-list + range scan |
| Q3 | Location + price category | Multi-column co-locality |
| Q4 | Full aggregation by brand | Scan-heavy analytics |
| Q5 | Year range + fuel_type | Partition key home turf |

In [0]:
# ── BENCHMARKING ENGINE ───────────────────────────────────────────────────────
def benchmark(query, label, runs=3):
    times = []
    for i in range(runs):
        start   = time.perf_counter()
        spark.sql(query).collect()
        elapsed = round(time.perf_counter() - start, 3)
        times.append(elapsed)
        print(f'      Run {i+1}: {elapsed}s')
    avg = round(sum(times) / len(times), 3)
    print(f'    [{label}] Average: {avg}s')
    return avg

# ── 5 QUERY PATTERNS ─────────────────────────────────────────────────────────
QUERIES = [
    (
        'Q1: Single brand filter',
        f'SELECT listing_id,brand,model,price_rub FROM {FACT_LIQUID} WHERE brand = "toyota"',
        f'SELECT listing_id,brand,model,price_rub FROM {FACT_PART}   WHERE brand = "toyota"',
    ),
    (
        'Q2: Multi-brand + mileage range',
        f'SELECT brand,model,COUNT(*),AVG(price_rub) FROM {FACT_LIQUID} WHERE brand IN ("toyota","honda","kia") AND mileage_km < 50000 GROUP BY brand,model',
        f'SELECT brand,model,COUNT(*),AVG(price_rub) FROM {FACT_PART}   WHERE brand IN ("toyota","honda","kia") AND mileage_km < 50000 GROUP BY brand,model',
    ),
    (
        'Q3: Location + price category',
        f'SELECT location_key,price_category,COUNT(*),AVG(price_usd) FROM {FACT_LIQUID} WHERE price_category = "LUXURY" GROUP BY location_key,price_category',
        f'SELECT location_key,price_category,COUNT(*),AVG(price_usd) FROM {FACT_PART}   WHERE price_category = "LUXURY" GROUP BY location_key,price_category',
    ),
    (
        'Q4: Full aggregation by brand (scan-heavy)',
        f'SELECT brand,COUNT(*) AS listings,AVG(price_rub) AS avg_price FROM {FACT_LIQUID} GROUP BY brand ORDER BY listings DESC',
        f'SELECT brand,COUNT(*) AS listings,AVG(price_rub) AS avg_price FROM {FACT_PART}   GROUP BY brand ORDER BY listings DESC',
    ),
    (
        'Q5: Year range + fuel_type (partition advantage)',
        f'SELECT listing_year,fuel_type,COUNT(*),SUM(price_rub) FROM {FACT_LIQUID} WHERE listing_year BETWEEN 2020 AND 2023 AND fuel_type = "Бензин" GROUP BY listing_year,fuel_type',
        f'SELECT listing_year,fuel_type,COUNT(*),SUM(price_rub) FROM {FACT_PART}   WHERE listing_year BETWEEN 2020 AND 2023 AND fuel_type = "Бензин" GROUP BY listing_year,fuel_type',
    ),
]

# ── EXECUTE ───────────────────────────────────────────────────────────────────
benchmark_results = []
print(f'  BENCHMARKING {len(QUERIES)} QUERY PATTERNS (3 runs each)')
print('=' * 70)

for label, q_liq, q_part in QUERIES:
    print(f'\n  {label}')
    print('  Liquid Clustering:')
    t_liq  = benchmark(q_liq,  'Liquid',  runs=3)
    print('  Partitioned + Z-Order:')
    t_part = benchmark(q_part, 'Z-Order', runs=3)
    winner  = 'Liquid' if t_liq <= t_part else 'Z-Order'
    speedup = round(max(t_liq, t_part) / min(t_liq, t_part), 2) if min(t_liq, t_part) > 0 else 1.0
    print(f'  Winner: {winner}  ({speedup}x faster)')
    benchmark_results.append((label, float(t_liq), float(t_part), winner, speedup))

print('\n  BENCHMARK COMPLETE')

In [0]:
# ── SAVE BENCHMARK RESULTS TO GOLD LAYER ────────────────────────────────────
schema = 'query STRING, liquid_avg_secs DOUBLE, zorder_avg_secs DOUBLE, winner STRING, speedup DOUBLE'
spark.createDataFrame(benchmark_results, schema) \
    .write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(BENCH_TABLE)

spark.sql(f'COMMENT ON TABLE {BENCH_TABLE} IS "Benchmark: Liquid vs Partitioned+Z-Order, 5 queries x 3 runs"')
print('  Benchmark results saved.')

display(spark.table(BENCH_TABLE).orderBy('query'))

liquid_wins = sum(1 for r in benchmark_results if r[3] == 'Liquid')
zorder_wins = len(benchmark_results) - liquid_wins
print(f'\n  Liquid wins: {liquid_wins}/{len(benchmark_results)}')
print(f'  Z-Order wins: {zorder_wins}/{len(benchmark_results)}')

## Section 4 — MERGE INTO: Late-Arriving Data Upserts

In [0]:
# ── BUILD LATE-ARRIVING BATCH ────────────────────────────────────────────────
before_count = spark.table(FACT_LIQUID).count()
print(f'   Rows BEFORE merge: {before_count:,}')

# 500 existing rows with 5% price revision + 10 new synthetic inserts
spark.sql(f'''
CREATE OR REPLACE TEMP VIEW late_arriving_batch AS

SELECT * FROM (
    SELECT
        listing_id,
        listing_date,
        brand, model, manufacture_year,
        ROUND(price_rub * 0.95, 2)          AS price_rub,
        ROUND(price_rub * 0.95 / 82.5, 2)  AS price_usd,
        price_category, car_age_years, car_age_at_listing,
        is_high_mileage, price_per_hp_usd,
        location_key, fuel_type, transmission_type, drive_type,
        steering_wheel, trim_level, has_license, engine_power, mileage_km,
        color_r, color_g, color_b, photo_count,
        listing_year, listing_month,
        bronze_load_dt, bronze_source_file, silver_load_dt,
        current_timestamp() AS gold_load_dt
    FROM {FACT} 
    LIMIT 500
)

UNION ALL

SELECT * FROM (
    SELECT
        CONCAT("NEW_", CAST(rn AS STRING)) AS listing_id,
        current_timestamp()                AS listing_date,
        brand, model, manufacture_year,
        price_rub, price_usd, price_category,
        car_age_years, car_age_at_listing, is_high_mileage, price_per_hp_usd,
        location_key, fuel_type, transmission_type, drive_type,
        steering_wheel, trim_level, has_license, engine_power, mileage_km,
        color_r, color_g, color_b, photo_count,
        listing_year, listing_month,
        bronze_load_dt, bronze_source_file, silver_load_dt,
        current_timestamp() AS gold_load_dt
    FROM (
        SELECT *, ROW_NUMBER() OVER (ORDER BY listing_id) AS rn
        FROM {FACT} 
        LIMIT 10
    )
)
''')

batch_cnt = spark.sql('SELECT COUNT(*) AS c FROM late_arriving_batch').collect()[0]['c']
print(f'   Late-arriving batch: {batch_cnt:,} rows (500 price revisions + 10 new inserts)')

In [0]:
# ── EXECUTE MERGE ─────────────────────────────────────────────────────────────
print('  Executing MERGE INTO fact_listings_liquid ...')

spark.sql(f'''
MERGE INTO {FACT_LIQUID} AS target
USING late_arriving_batch AS source
ON target.listing_id = source.listing_id

WHEN MATCHED AND target.price_rub != source.price_rub THEN
  UPDATE SET
    target.price_rub      = source.price_rub,
    target.price_usd      = source.price_usd,
    target.price_category = CASE
        WHEN source.price_rub < 300000               THEN "BUDGET"
        WHEN source.price_rub BETWEEN 300000 AND 700000   THEN "MID_RANGE"
        WHEN source.price_rub BETWEEN 700001 AND 1500000  THEN "PREMIUM"
        WHEN source.price_rub > 1500000              THEN "LUXURY"
        ELSE "UNKNOWN" END,
    target.gold_load_dt   = source.gold_load_dt

WHEN NOT MATCHED THEN
  INSERT (
    listing_id, listing_date, brand, model, manufacture_year,
    price_rub, price_usd, price_category,
    car_age_years, car_age_at_listing, is_high_mileage, price_per_hp_usd,
    location_key, fuel_type, transmission_type, drive_type,
    steering_wheel, trim_level, has_license, engine_power, mileage_km,
    color_r, color_g, color_b, photo_count,
    listing_year, listing_month,
    bronze_load_dt, bronze_source_file, silver_load_dt, gold_load_dt
  )
  VALUES (
    source.listing_id, source.listing_date, source.brand, source.model, source.manufacture_year,
    source.price_rub, source.price_usd, source.price_category,
    source.car_age_years, source.car_age_at_listing, source.is_high_mileage, source.price_per_hp_usd,
    source.location_key, source.fuel_type, source.transmission_type, source.drive_type,
    source.steering_wheel, source.trim_level, source.has_license, source.engine_power, source.mileage_km,
    source.color_r, source.color_g, source.color_b, source.photo_count,
    source.listing_year, source.listing_month,
    source.bronze_load_dt, source.bronze_source_file, source.silver_load_dt, source.gold_load_dt
  )
''')

after_count = spark.table(FACT_LIQUID).count()
new_rows    = after_count - before_count
print(f'  Rows BEFORE: {before_count:,}')
print(f'  Rows AFTER : {after_count:,}')
print(f'  Net new    : {new_rows:,} (expected ~10 inserts)')

print('\n  Sample of updated rows:')
display(spark.sql(f'SELECT listing_id,brand,price_rub,price_usd,gold_load_dt FROM {FACT_LIQUID} WHERE listing_id LIKE "NEW_%" LIMIT 5'))

## Section 5 — Churn Metrics


**Two churn tables:**
1. `agg_stale_inventory` — grain: brand+model+location, with churn_status label
2. `agg_brand_churn_metrics` — grain: brand, with churn_score_pct (% of combos gone stale)

In [0]:
# ── CHURN METRIC 1: STALE INVENTORY ─────────────────────────────────────────
print('  Building Stale Inventory churn metric (>180 days inactive) ...')

spark.sql(f'''
CREATE OR REPLACE TABLE {CHURN_TABLE}
USING DELTA
TBLPROPERTIES (
  'quality'           = 'gold',
  'metric_type'       = 'churn',
  'churn_threshold'   = '180_days',
  'churn_grain'       = 'brand_model_location',
  'refresh_frequency' = 'daily'
)
AS
WITH daily_activity AS (
    SELECT
        brand, model, location_key, fuel_type,
        CAST(listing_date AS DATE)  AS listing_day,
        COUNT(*)                    AS daily_listings,
        AVG(price_rub)              AS avg_price_rub,
        AVG(price_usd)              AS avg_price_usd
    FROM {FACT_LIQUID}
    WHERE listing_date IS NOT NULL
    GROUP BY brand, model, location_key, fuel_type, CAST(listing_date AS DATE)
),
last_seen AS (
    SELECT
        brand, model, location_key, fuel_type,
        MAX(listing_day)            AS last_listing_date,
        MIN(listing_day)            AS first_listing_date,
        COUNT(DISTINCT listing_day) AS active_days,
        SUM(daily_listings)         AS total_listings,
        ROUND(AVG(avg_price_rub),0) AS avg_price_rub,
        ROUND(AVG(avg_price_usd),2) AS avg_price_usd
    FROM daily_activity
    GROUP BY brand, model, location_key, fuel_type
)
SELECT
    brand, model, location_key, fuel_type,
    last_listing_date,
    first_listing_date,
    active_days,
    total_listings,
    avg_price_rub,
    avg_price_usd,
    DATEDIFF(current_date(), last_listing_date) AS days_inactive,
    CASE
        WHEN DATEDIFF(current_date(), last_listing_date) > 365 THEN 'CRITICAL'
        WHEN DATEDIFF(current_date(), last_listing_date) > 180 THEN 'STALE'
        WHEN DATEDIFF(current_date(), last_listing_date) > 90  THEN 'AT_RISK'
        ELSE 'ACTIVE'
    END AS churn_status,
    current_timestamp() AS metric_computed_at
FROM last_seen
WHERE DATEDIFF(current_date(), last_listing_date) > 180
''')

spark.sql(f'COMMENT ON TABLE {CHURN_TABLE} IS "Churn: brand+model+location combos inactive >180 days"')

stale_cnt = spark.table(CHURN_TABLE).count()
print(f'  Stale records: {stale_cnt:,} brand+model+location combos')

print('\n  Churn status breakdown:')
# Make sure 'from pyspark.sql import functions as F' is imported
spark.table(CHURN_TABLE).groupBy('churn_status').count().orderBy(F.desc('count')).show(truncate=False)

In [0]:
# ── CHURN METRIC 2: BRAND CHURN SCORE ────────────────────────────────────────
print('  Building Brand Churn Score metric ...')

spark.sql(f'''
CREATE OR REPLACE TABLE {BRAND_CHURN}
USING DELTA
TBLPROPERTIES (
  'quality'           = 'gold',
  'metric_type'       = 'churn',
  'churn_grain'       = 'brand',
  'refresh_frequency' = 'daily'
)
AS
WITH brand_totals AS (
    SELECT
        brand,
        COUNT(DISTINCT CONCAT(model, "_", location_key)) AS total_combos,
        COUNT(DISTINCT model)                             AS distinct_models,
        COUNT(DISTINCT location_key)                      AS distinct_locations,
        COUNT(*)                                          AS total_listings,
        ROUND(AVG(price_rub), 0)                          AS avg_price_rub,
        ROUND(AVG(mileage_km), 0)                         AS avg_mileage_km,
        MAX(CAST(listing_date AS DATE))                   AS brand_last_seen
    FROM {FACT_LIQUID}
    WHERE listing_date IS NOT NULL
    GROUP BY brand
),
brand_stale AS (
    SELECT
        brand,
        COUNT(*)                    AS stale_combos,
        ROUND(AVG(days_inactive),0) AS avg_days_inactive,
        MAX(days_inactive)          AS max_days_inactive
    FROM {CHURN_TABLE}
    GROUP BY brand
)
SELECT
    t.brand,
    t.total_combos,
    COALESCE(s.stale_combos, 0)                                           AS stale_combos,
    t.distinct_models,
    t.distinct_locations,
    t.total_listings,
    t.avg_price_rub,
    t.avg_mileage_km,
    t.brand_last_seen,
    COALESCE(s.avg_days_inactive, 0)                                      AS avg_days_inactive,
    COALESCE(s.max_days_inactive, 0)                                      AS max_days_inactive,
    ROUND(COALESCE(s.stale_combos,0) / t.total_combos * 100.0, 1)        AS churn_score_pct,
    CASE
        WHEN ROUND(COALESCE(s.stale_combos,0)/t.total_combos*100.0,1) > 80 THEN 'CRITICAL'
        WHEN ROUND(COALESCE(s.stale_combos,0)/t.total_combos*100.0,1) > 50 THEN 'HIGH'
        WHEN ROUND(COALESCE(s.stale_combos,0)/t.total_combos*100.0,1) > 20 THEN 'MEDIUM'
        ELSE 'LOW'
    END AS churn_risk,
    current_timestamp() AS metric_computed_at
FROM brand_totals t
LEFT JOIN brand_stale s ON t.brand = s.brand
ORDER BY churn_score_pct DESC
''')

spark.sql(f"COMMENT ON TABLE {BRAND_CHURN} IS 'Brand churn score: pct of model+location combos inactive >180 days'")

brand_cnt = spark.table(BRAND_CHURN).count()
print(f'  Brand churn metrics: {brand_cnt:,} brands scored')

print('\n  Brand churn risk distribution:')
spark.table(BRAND_CHURN).groupBy('churn_risk').count().orderBy(F.desc('count')).show(truncate=False)

## Section 6 — Delta Maintenance

- **VACUUM** — removes unreferenced old files. 168-hour retention = 7-day time travel window.
- **OPTIMIZE** — compacts small files. On Liquid tables, also re-applies clustering.

In [0]:
# ── VACUUM ────────────────────────────────────────────────────────────────────
print('  VACUUM (168-hour retention) ...')
for tbl in [FACT_LIQUID, FACT_PART, CHURN_TABLE, BRAND_CHURN]:
    spark.sql(f'VACUUM {tbl} RETAIN 168 HOURS')
    print(f'  VACUUM done: {tbl.split(".")[-1]}')

# ── OPTIMIZE ──────────────────────────────────────────────────────────────────
print('\n  OPTIMIZE ...')
spark.sql(f'OPTIMIZE {FACT_LIQUID}')                               # re-clusters
spark.sql(f'OPTIMIZE {FACT_PART} ZORDER BY (brand, listing_year)') # re-z-orders
print('  OPTIMIZE complete')

# ── HISTORY ───────────────────────────────────────────────────────────────────
print('\n  Delta History (fact_listings_liquid, last 8 ops):')
display(
    spark.sql(f'DESCRIBE HISTORY {FACT_LIQUID}')
    .select('version','timestamp','operation','operationMetrics')
    .orderBy('version', ascending=False)
    .limit(8)
)

## Section 7 — Final Summary & Strategy Comparison

In [0]:
# ── FINAL SUMMARY ────────────────────────────────────────────────────────────
print('=' * 70)
print('  FINAL SUMMARY')
print('=' * 70)

tables_info = [
    (FACT,        'fact_listings (source)'),
    (FACT_LIQUID, 'fact_listings_liquid (Liquid Clustering)'),
    (FACT_PART,   'fact_listings_partitioned (Partition+Z-Order)'),
    (CHURN_TABLE, 'agg_stale_inventory (Churn Metric 1)'),
    (BRAND_CHURN, 'agg_brand_churn_metrics (Churn Metric 2)'),
    (BENCH_TABLE, 'benchmark_results'),
]
print()
for tbl, label in tables_info:
    cnt = spark.table(tbl).count()
    print(f'  {label:<50} {cnt:>10,} rows')

print('\n  Benchmark Results:')
display(spark.table(BENCH_TABLE).orderBy('query'))

print()
total_stale  = spark.table(CHURN_TABLE).count()
critical_cnt = spark.table(CHURN_TABLE).filter("churn_status = 'CRITICAL'").count()
print(f'  Stale inventory combos: {total_stale:,}')
print(f'  CRITICAL (>365 days)  : {critical_cnt:,}')

print()
print('  Brand churn risk distribution:')
spark.table(BRAND_CHURN).groupBy('churn_risk').count().orderBy(F.desc('count')).show(truncate=False)

print('=' * 70)
print('  STRATEGY RECOMMENDATION')
print('=' * 70)
print()
print('  USE Liquid Clustering when:')
print('    Filters span multiple columns (brand + location + year)')
print('    Filter combinations are unpredictable (BI / ad-hoc)')
print('    You want zero partition management overhead')
print()
print('  USE Partitioned + Z-Order when:')
print('    One column is ALWAYS in the WHERE clause (e.g. fuel_type)')
print('    Very high data volume per partition')
print()
